# Colab Evaluation Workflow

Run baseline and LoRA generation, then score TIFA and GenAI-Bench with API judges.

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content
!git clone https://github.com/BoatHermit/T2I-RL-Eval.git
%cd /content/T2I-RL-Eval
!pip install -r requirements.txt

/content
Cloning into 'T2I-RL-Eval'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 187 (delta 84), reused 158 (delta 55), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 28.17 MiB | 28.90 MiB/s, done.
Resolving deltas: 100% (84/84), done.
/content/T2I-RL-Eval
  Cloning https://github.com/deepseek-ai/Janus.git to /tmp/pip-install-pljw6b1r/janus_e1e7a847edd646e2aba49df97b77cbb8
  Running command git clone --filter=blob:none --quiet https://github.com/deepseek-ai/Janus.git /tmp/pip-install-pljw6b1r/janus_e1e7a847edd646e2aba49df97b77cbb8
  Resolved https://github.com/deepseek-ai/Janus.git to commit 1daa72fa409002d40931bd7b36a9280362469ead
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## 2. Configuration

In [ ]:
import os

API_KEY="sk-XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
API_BASE_URL="https://api.siliconflow.cn/v1"
JUDGE_MODEL="Qwen/Qwen3.5-9B"

os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_BASE_URL"] = API_BASE_URL
os.environ["JUDGE_MODEL"] = JUDGE_MODEL

BASE_MODEL = "deepseek-ai/Janus-Pro-1B"
LORA_DIR = "artifacts/lora/grpo_siliconflow_quick_final"
OUTPUT_DIR = "/content/drive/MyDrive/cuhk_courses/aims5740/evaluation"

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/T2I-RL-Eval")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 3. Generate Before Images

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant before \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 72

/content/T2I-RL-Eval
[generate] benchmark=tifa variant=before samples=4073 prompt_batch_size=72 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-10 07:02:44.863064: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 07:02:44.871617: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775804564.881342    1898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775804564.884612    1898 cuda_blas.cc:1407] Unable to register cuBLAS fac

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant before \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 72

/content/T2I-RL-Eval
[generate] benchmark=genai_bench variant=before samples=1600 prompt_batch_size=72 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-10 07:20:05.294107: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 07:20:05.302863: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775805605.312841    6671 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775805605.316155    6671 cuda_blas.cc:1407] Unable to register cuB

## 4. Generate After Images

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant after \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 36

/content/T2I-RL-Eval
[generate] benchmark=tifa variant=after samples=4073 prompt_batch_size=36 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-10 07:26:50.372517: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 07:26:50.381362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775806010.391326    8585 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775806010.394648    8585 cuda_blas.cc:1407] Unable to register cuBLAS fact

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant after \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 36

/content/T2I-RL-Eval
[generate] benchmark=genai_bench variant=after samples=1600 prompt_batch_size=36 dtype=float16 device=cuda
[generate] loading model=deepseek-ai/Janus-Pro-1B
Python version is above 3.10, patching the collections module.
2026-04-10 07:51:37.795621: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-10 07:51:37.804469: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775807497.814388   14945 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775807497.817749   14945 cuda_blas.cc:1407] Unable to register cuBL

## 5. Run TIFA

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/run_tifa.py \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --images_root $OUTPUT_DIR/images \
        --variant before \
        --output_path $OUTPUT_DIR/results/tifa_before.jsonl \
        --resume \
        --max_workers 8 \
        --log_every 1
!python scripts/run_tifa.py \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --images_root $OUTPUT_DIR/images \
        --variant after \
        --output_path $OUTPUT_DIR/results/tifa_after.jsonl \
        --resume \
        --max_workers 8 \
        --log_every 1

Streaming output truncated to the last 5000 lines.
[tifa] completed=2663/3585 last_sample=partiprompt_997 rate=0.75 samples/s elapsed=59.2 min
[tifa] completed=2664/3585 last_sample=partiprompt_991 rate=0.75 samples/s elapsed=59.2 min
[tifa] completed=2665/3585 last_sample=partiprompt_996 rate=0.75 samples/s elapsed=59.2 min
[tifa] completed=2666/3585 last_sample=partiprompt_998 rate=0.75 samples/s elapsed=59.2 min
[tifa] completed=2667/3585 last_sample=partiprompt_999 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2668/3585 last_sample=partiprompt_1001 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2669/3585 last_sample=partiprompt_1000 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2670/3585 last_sample=partiprompt_1002 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2671/3585 last_sample=partiprompt_1003 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2672/3585 last_sample=partiprompt_1005 rate=0.75 samples/s elapsed=59.3 min
[tifa] completed=2673/35

## 6. Run GenAI-Bench

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/run_genai_bench.py \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --images_root $OUTPUT_DIR/images \
        --variant before \
        --output_path $OUTPUT_DIR/results/genai_before.jsonl \
        --resume \
        --max_workers 8 \
        --log_every 1
!python scripts/run_genai_bench.py \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --images_root $OUTPUT_DIR/images \
        --variant after \
        --output_path $OUTPUT_DIR/results/genai_after.jsonl \
        --resume \
        --max_workers 8 \
        --log_every 1

/content/T2I-RL-Eval
[genai] manifest=data/evaluation/genai_bench/samples.jsonl variant=before output=/content/drive/MyDrive/cuhk_courses/aims5740/evaluation/results/genai_before.jsonl judge_model=Qwen/Qwen3.5-9B base_url=https://api.siliconflow.cn/v1 workers=8
[genai] variant=before total=1600 pending=0 skipped=1600 workers=8
[genai] nothing to do for variant=before
[genai] manifest=data/evaluation/genai_bench/samples.jsonl variant=after output=/content/drive/MyDrive/cuhk_courses/aims5740/evaluation/results/genai_after.jsonl judge_model=Qwen/Qwen3.5-9B base_url=https://api.siliconflow.cn/v1 workers=8
[genai] variant=after total=1600 pending=1453 skipped=147 workers=8
[genai] completed=1/1453 last_sample=00153 rate=0.10 samples/s elapsed=0.2 min
[genai] completed=2/1453 last_sample=00149 rate=0.21 samples/s elapsed=0.2 min
[genai] completed=3/1453 last_sample=00147 rate=0.31 samples/s elapsed=0.2 min
[genai] completed=4/1453 last_sample=00151 rate=0.42 samples/s elapsed=0.2 min
[genai]

## 7. Summary

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/summarize_evaluation.py \
        --tifa_results_before $OUTPUT_DIR/results/tifa_before.jsonl \
        --tifa_results_after $OUTPUT_DIR/results/tifa_after.jsonl \
        --genai_results_before $OUTPUT_DIR/results/genai_before.jsonl \
        --genai_results_after $OUTPUT_DIR/results/genai_after.jsonl \
        --output_dir $OUTPUT_DIR/reports

/content/T2I-RL-Eval
